# Solution 04 — consumer groups & manual offsets

Reference for the manual-commit pattern, the two-consumer experiment, creating a topic with `rpk` and the replay. The interesting bit isn't the code; it's the *behaviour* you should observe.

In [ ]:
from confluent_kafka import Consumer

BROKER = 'redpanda:29092'

consumer = Consumer({
    'bootstrap.servers':  BROKER,
    'group.id':           'manual-commit-demo',
    'auto.offset.reset':  'earliest',
    'enable.auto.commit': False,
})
consumer.subscribe(['rider-locations'])

messages_read, empty_polls = 0, 0
while messages_read < 5 and empty_polls < 5:
    msg = consumer.poll(2.0)
    if msg is None: empty_polls += 1; continue
    if msg.error(): continue
    empty_polls = 0; messages_read += 1
    print(f'#{messages_read} offset={msg.offset()} -- processing...', end='')
    consumer.commit(message=msg)
    print(' committed.')
consumer.close()

## Task A — observing re-delivery without commit

We deliberately *don't* commit. Run this twice in a row with the same `group.id` and you'll see the same offsets pop up both times.

In [ ]:
consumer = Consumer({
    'bootstrap.servers':  BROKER,
    'group.id':           'no-commit-demo',
    'auto.offset.reset':  'earliest',
    'enable.auto.commit': False,
})
consumer.subscribe(['rider-locations'])

messages_read, empty_polls = 0, 0
while messages_read < 3 and empty_polls < 5:
    msg = consumer.poll(2.0)
    if msg is None: empty_polls += 1; continue
    if msg.error(): continue
    empty_polls = 0; messages_read += 1
    print(f'#{messages_read} P{msg.partition()} offset={msg.offset()} -- (NOT committing)')
consumer.close()

## Task B — two consumers in one group

Run this cell in two tabs and produce events from a third. Each tab only prints the partitions it owns. When you stop one tab, the group rebalances and the other tab takes over all partitions after a few seconds.

In [ ]:
from confluent_kafka import Consumer
from datetime import datetime
import time

BROKER = 'redpanda:29092'
LISTEN_SECONDS = 300   # stops by itself after 5 minutes; stop earlier with ■

consumer = Consumer({
    'bootstrap.servers': BROKER,
    'group.id':          'group-experiment',
    'auto.offset.reset': 'latest',         # only NEW events from now
})
consumer.subscribe(['rider-locations', 'orders'])
print('Listening — start the producer in another tab. Stop with the ■ button.')

deadline = time.time() + LISTEN_SECONDS
try:
    while time.time() < deadline:
        msg = consumer.poll(0.5)
        if msg is None or msg.error(): continue
        ts = datetime.now().strftime('%H:%M:%S')
        key = msg.key().decode() if msg.key() else 'None'
        print(f'[{ts}] {msg.topic()} P{msg.partition()} '
              f'offset={msg.offset()} key={key}')
    print('Time is up.')
except KeyboardInterrupt:
    print('\nStopped.')
finally:
    consumer.close()

## Task C — create a topic yourself

In the terminal (1 day = 24 × 60 × 60 × 1000 ms = `86400000`):

```bash
rpk topic create order-replay --partitions 3 --replicas 1 \
    --topic-config retention.ms=86400000 \
    -X brokers=redpanda:29092
rpk topic describe order-replay -X brokers=redpanda:29092
rpk topic describe order-replay -p -X brokers=redpanda:29092
```

The `SUMMARY` shows `PARTITIONS 3` and `REPLICAS 1`; `-p` lists the three partitions with their offsets. In the `CONFIGS` section:

```
retention.bytes   -1         DEFAULT_CONFIG
retention.ms      86400000   DYNAMIC_TOPIC_CONFIG
```

- `DYNAMIC_TOPIC_CONFIG` means: set for this topic, overriding the broker default (7 days). `retention.bytes = -1` means no size limit: only age counts.
- **Replicas 1:** the Codespace has a single broker, so there is nowhere to put a copy. With three brokers you'd choose 3, so the topic survives the loss of a broker.
- After one day the broker deletes the event (in whole log segments, so it can stay a bit longer). Consumers that haven't read it by then will never see it, and a replay can only go back as far as the retention.

In [ ]:
from confluent_kafka import Producer
import json, time, random

BROKER = 'redpanda:29092'
TOPIC  = 'order-replay'

producer  = Producer({'bootstrap.servers': BROKER})
customers = ['Anna', 'Ben', 'Clara', 'David', 'Elif']


def send_orders(first, last):
    """Send the orders order-<first> ... order-<last> with status 'placed'."""
    for n in range(first, last + 1):
        order_id = f'order-{n}'
        event = {
            'order_id':   order_id,
            'customer':   random.choice(customers),
            'restaurant': 'Pizzeria Roma',
            'total_chf':  round(random.uniform(15, 50), 2),
            'status':     'placed',
            'timestamp':  time.time(),
        }
        producer.produce(TOPIC, key=order_id, value=json.dumps(event))
    producer.flush()


send_orders(5001, 5005)
time.sleep(3)
MIDDLE = int(time.time())   # remember this moment (UNIX seconds) for the replay by time
time.sleep(3)
send_orders(5006, 5010)
print(f'Sent 10 orders. order-5006 ... order-5010 came after MIDDLE = {MIDDLE}')

## Task D — replay

### Step 1 — read everything once

The first run prints 10 orders. The second run prints `0 orders read`: on `close()` the consumer committed its position (auto-commit is on), and the group continues from there. `auto.offset.reset` only applies to a group **without** a committed offset.

In [ ]:
from confluent_kafka import Consumer


def read_orders(group_id='replay-demo'):
    """Read everything the group has not seen yet, then close (commits the position)."""
    consumer = Consumer({
        'bootstrap.servers': BROKER,
        'group.id':          group_id,
        'auto.offset.reset': 'earliest',   # a NEW group starts at the beginning
    })
    consumer.subscribe([TOPIC])
    read, empty_polls = 0, 0
    try:
        while empty_polls < 5:
            msg = consumer.poll(2.0)
            if msg is None: empty_polls += 1; continue
            if msg.error(): continue
            empty_polls = 0; read += 1
            print(f'P{msg.partition()} offset={msg.offset()} key={msg.key().decode()}')
    finally:
        consumer.close()   # commits the last position and leaves the group
    print(f'{read} orders read with group {group_id!r}')


read_orders()

### Step 2 — move the group back

```bash
rpk group describe replay-demo -X brokers=redpanda:29092
rpk group seek replay-demo --to start -X brokers=redpanda:29092
rpk group describe replay-demo -X brokers=redpanda:29092
```

Before the seek: `STATE Empty`, `CURRENT-OFFSET` = `LOG-END-OFFSET`, `LAG 0`. The seek prints `PRIOR-OFFSET` and `CURRENT-OFFSET` per partition. After it, `CURRENT-OFFSET` is `0` and `LAG` equals the number of events in each partition — `TOTAL-LAG 10`. Reading again brings back all 10 orders:

In [ ]:
read_orders()   # all 10 orders again

### Step 3 — replay from a point in time

`rpk group seek replay-demo --to <MIDDLE>` moves every partition to the first event whose timestamp is at or after `MIDDLE` (seconds, milliseconds or nanoseconds are all accepted). Only `order-5006` … `order-5010` come back. Kafka stores a timestamp with every event (by default the producer's send time), which is what the seek searches — not the `timestamp` field inside our JSON.

In [ ]:
print(MIDDLE)
# in the terminal: rpk group seek replay-demo --to <MIDDLE> -X brokers=redpanda:29092
read_orders()   # only order-5006 ... order-5010

### Step 4 — why the group must be inactive

While the cell below runs, the seek fails:

```
TOPIC         PARTITION  PRIOR-OFFSET  CURRENT-OFFSET  ERROR
order-replay  0          -1            -1              INVALID_OPERATION: seeking a non-empty group is not allowed.
```

and `rpk group describe` shows `STATE Stable`, `MEMBERS 1`.

In [ ]:
import time

consumer = Consumer({
    'bootstrap.servers': BROKER,
    'group.id':          'replay-demo',
    'auto.offset.reset': 'earliest',
})
consumer.subscribe([TOPIC])
print('Member of the group replay-demo for 60 s -- run the seek in the terminal now.')
deadline = time.time() + 60
try:
    while time.time() < deadline:
        consumer.poll(0.5)
finally:
    consumer.close()
print('Left the group.')

**Why?** The running consumer keeps its current position in memory and commits it again every few seconds (auto-commit), or after every message (manual commit). A position written from outside would be overwritten right away — or worse, half of the partitions would be at the new and half at the old position. So the broker only lets you reset a group without members (`STATE Empty`).

The order for a replay in production: **stop** all consumers of the group → **seek** → **start** them again. (Note that this cell also read and committed the orders: the group is at the end again.)

### Step 5 — replay from Python

`offsets_for_times()` translates the timestamp into an offset per partition, `assign()` starts reading there. Because the consumer uses `assign()` and never commits, it does not touch any group's committed position and works even while other consumers are running. This is handy for a one-off look at the past; for re-processing with a service the group seek is the usual way.

In [ ]:
from confluent_kafka import Consumer, TopicPartition

consumer = Consumer({
    'bootstrap.servers':  BROKER,
    'group.id':           'replay-python',
    'enable.auto.commit': False,
})

# One entry per partition; the "offset" field carries the timestamp in MILLISECONDS.
wanted = [TopicPartition(TOPIC, p, MIDDLE * 1000) for p in range(3)]

# The broker looks up the first offset whose timestamp is >= MIDDLE, per partition.
offsets = consumer.offsets_for_times(wanted, timeout=10)
for tp in offsets:
    print(f'partition {tp.partition}: start at offset {tp.offset}')   # -1 = nothing after MIDDLE
# assign() instead of subscribe(): we pick partitions and offsets ourselves,
# no group rebalancing, and the committed offsets of the group stay untouched.
consumer.assign(offsets)

read, empty_polls = 0, 0
while empty_polls < 3:
    msg = consumer.poll(2.0)
    if msg is None: empty_polls += 1; continue
    if msg.error(): continue
    empty_polls = 0; read += 1
    print(f'P{msg.partition()} offset={msg.offset()} key={msg.key().decode()}')
consumer.close()
print(f'{read} orders read from MIDDLE on')